# AIND Ephys Dispatch

Build an `AINDEPhysDispatchScanConfig`, expand it with `GridScanGenerationTask`, run the `AINDEPhysDispatchTask` for each coordinate to produce a CLI command, then actually invoke the dispatch script ([aind-ephys-job-dispatch/code/run](https://github.com/AllenNeuralDynamics/aind-ephys-job-dispatch/blob/main/code/run)) on the toy recording.

This notebook targets the toy SpikeInterface recording produced by `0_generate_toy_recording.ipynb` (saved at `./toy_example_recording`). It is a `BinaryFolderRecording`, so we use the `spikeinterface` input mode with `reader_type='binaryfolder'`.

## Imports

In [1]:
import json
import shlex
import subprocess
from pathlib import Path

import obi_one as obi
from obi_one.scientific.tasks.spike_sorting.dispatch.blocks import (
    DispatchBasic,
    DispatchDataDependent,
    DispatchDebug,
    SpikeInterfaceInfo,
)

## Build the scan config

Any field set to a list becomes a dimension of the parameter scan. Below we sweep `min_recording_duration` (2 values) and `debug_mode` (2 values), giving a 2x2 = 4 coordinate scan.

The recording is loaded by SpikeInterface via the `binaryfolder` extractor, pointing at the saved toy recording with an absolute path.

In [2]:
recording_folder = (Path.cwd() / "toy_example_recording").resolve()
assert recording_folder.exists(), (
    f"{recording_folder} not found. Run 0_generate_toy_recording.ipynb first."
)

scan_config = obi.AINDEPhysDispatchScanConfig(
    dispatch_basic=DispatchBasic(
        split_segments=True,
        split_groups=True,
        skip_timestamps_check=False,
        min_recording_duration=[1.0, 5.0],
    ),
    dispatch_data_dependent=DispatchDataDependent(
        input_format="spikeinterface",
        multi_session_data=False,
        spikeinterface_info=SpikeInterfaceInfo(
            reader_type="binaryfolder",
            reader_kwargs={"folder_path": str(recording_folder)},
        ),
    ),
    dispatch_debug=DispatchDebug(
        debug_mode=[True, False],
        debug_duration=10.0,
    ),
)

## Generate the grid scan and run the task for each coordinate

In [3]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/aind_ephys_dispatch/grid_scan",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)

[2026-04-29 14:23:58,508] INFO: python -u code/run_capsule.py --input spikeinterface --min-recording-duration 1.0 --debug --debug-duration 10.0 --spikeinterface-info '{"reader_type": "binaryfolder", "reader_kwargs": {"folder_path": "/Users/james/Documents/obi/code/obi-main/obi-one/examples/K_extracellular/toy_example_recording"}}'


[2026-04-29 14:23:58,508] INFO: python -u code/run_capsule.py --input spikeinterface --min-recording-duration 1.0 --spikeinterface-info '{"reader_type": "binaryfolder", "reader_kwargs": {"folder_path": "/Users/james/Documents/obi/code/obi-main/obi-one/examples/K_extracellular/toy_example_recording"}}'


[2026-04-29 14:23:58,508] INFO: python -u code/run_capsule.py --input spikeinterface --min-recording-duration 5.0 --debug --debug-duration 10.0 --spikeinterface-info '{"reader_type": "binaryfolder", "reader_kwargs": {"folder_path": "/Users/james/Documents/obi/code/obi-main/obi-one/examples/K_extracellular/toy_example_recording"}}'


[2026-04-29 14:23:58,508] INFO: python -u code/run_capsule.py --input spikeinterface --min-recording-duration 5.0 --spikeinterface-info '{"reader_type": "binaryfolder", "reader_kwargs": {"folder_path": "/Users/james/Documents/obi/code/obi-main/obi-one/examples/K_extracellular/toy_example_recording"}}'


['python -u code/run_capsule.py --input spikeinterface --min-recording-duration 1.0 --debug --debug-duration 10.0 --spikeinterface-info \'{"reader_type": "binaryfolder", "reader_kwargs": {"folder_path": "/Users/james/Documents/obi/code/obi-main/obi-one/examples/K_extracellular/toy_example_recording"}}\'',
 'python -u code/run_capsule.py --input spikeinterface --min-recording-duration 1.0 --spikeinterface-info \'{"reader_type": "binaryfolder", "reader_kwargs": {"folder_path": "/Users/james/Documents/obi/code/obi-main/obi-one/examples/K_extracellular/toy_example_recording"}}\'',
 'python -u code/run_capsule.py --input spikeinterface --min-recording-duration 5.0 --debug --debug-duration 10.0 --spikeinterface-info \'{"reader_type": "binaryfolder", "reader_kwargs": {"folder_path": "/Users/james/Documents/obi/code/obi-main/obi-one/examples/K_extracellular/toy_example_recording"}}\'',
 'python -u code/run_capsule.py --input spikeinterface --min-recording-duration 5.0 --spikeinterface-info \'{

## Inspect the generated single configs

In [4]:
for single_config in grid_scan.single_configs:
    print("---")
    print("min_recording_duration:", single_config.dispatch_basic.min_recording_duration)
    print("debug_mode:            ", single_config.dispatch_debug.debug_mode)
    print("command:")
    print("  ", single_config.command_line_representation())

---
min_recording_duration: 1.0
debug_mode:             True
command:
   python -u code/run_capsule.py --input spikeinterface --min-recording-duration 1.0 --debug --debug-duration 10.0 --spikeinterface-info '{"reader_type": "binaryfolder", "reader_kwargs": {"folder_path": "/Users/james/Documents/obi/code/obi-main/obi-one/examples/K_extracellular/toy_example_recording"}}'
---
min_recording_duration: 1.0
debug_mode:             False
command:
   python -u code/run_capsule.py --input spikeinterface --min-recording-duration 1.0 --spikeinterface-info '{"reader_type": "binaryfolder", "reader_kwargs": {"folder_path": "/Users/james/Documents/obi/code/obi-main/obi-one/examples/K_extracellular/toy_example_recording"}}'
---
min_recording_duration: 5.0
debug_mode:             True
command:
   python -u code/run_capsule.py --input spikeinterface --min-recording-duration 5.0 --debug --debug-duration 10.0 --spikeinterface-info '{"reader_type": "binaryfolder", "reader_kwargs": {"folder_path": "/Users/

## Actually run the dispatch script

Clone (or reuse) the `aind-ephys-job-dispatch` repo, create the `data/` and `results/` folders the capsule expects (the script writes generated `job_*.json` files into `../results` relative to `code/run`), then invoke the command for the first single config of the scan.

The script's only required runtime deps are `spikeinterface`, `probeinterface`, and `numpy`, which are already available from `0_generate_toy_recording.ipynb`.

In [5]:
dispatch_repo = Path("/tmp/aind-ephys-job-dispatch")
if not dispatch_repo.exists():
    subprocess.run(
        [
            "git", "clone", "--depth=1",
            "https://github.com/AllenNeuralDynamics/aind-ephys-job-dispatch.git",
            str(dispatch_repo),
        ],
        check=True,
    )

(dispatch_repo / "data").mkdir(exist_ok=True)
results_dir = dispatch_repo / "results"
results_dir.mkdir(exist_ok=True)
for stale in results_dir.glob("job_*.json"):
    stale.unlink()

# Output folder next to this notebook where we'll copy the generated job files.
local_results_dir = Path.cwd() / "dispatch_results"
local_results_dir.mkdir(exist_ok=True)
for stale in local_results_dir.glob("job_*.json"):
    stale.unlink()

In [6]:
import shutil

single_config = grid_scan.single_configs[0]
command_str = single_config.command_line_representation()
print("Generated command (run from capsule root):")
print(" ", command_str)
print()

# The capsule resolves `../data` and `../results` relative to CWD, so we cd into
# `code/` and strip the `code/` prefix from the script path before invoking.
argv = shlex.split(command_str)
argv = [a.removeprefix("code/") for a in argv]
result = subprocess.run(
    argv,
    cwd=dispatch_repo / "code",
    capture_output=True,
    text=True,
    check=False,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:\n", result.stderr)
    raise RuntimeError(f"dispatch run failed with code {result.returncode}")

# Copy the generated job files into the notebook's `dispatch_results/` folder.
for jf in sorted(results_dir.glob("job_*.json")):
    shutil.copy2(jf, local_results_dir / jf.name)
print("Copied job files to:", local_results_dir)

Generated command (run from capsule root):
  python -u code/run_capsule.py --input spikeinterface --min-recording-duration 1.0 --debug --debug-duration 10.0 --spikeinterface-info '{"reader_type": "binaryfolder", "reader_kwargs": {"folder_path": "/Users/james/Documents/obi/code/obi-main/obi-one/examples/K_extracellular/toy_example_recording"}}'



Running job dispatcher with the following parameters:
	SPLIT SEGMENTS: True
	SPLIT GROUPS: True
	DEBUG: True
	DEBUG DURATION: 10.0
	SKIP TIMESTAMPS CHECK: False
	MULTI SESSION: False
	INPUT: spikeinterface
	MIN_RECORDING_DURATION: 1.0
	SPIKEINTERFACE_INFO: {"reader_type": "binaryfolder", "reader_kwargs": {"folder_path": "/Users/james/Documents/obi/code/obi-main/obi-one/examples/K_extracellular/toy_example_recording"}}
Parsing spikeinterface input folder
	Stream name: None
Recording to be processed in parallel:
Relative to: ../data
	block0_None_recording1
		Duration: 10.0 s - Num. channels: 70
Generated 1 job config files

Copied job files to: /Users/james/Documents/obi/code/obi-main/obi-one/examples/K_extracellular/dispatch_results


## Inspect the generated job JSON

The dispatcher writes one `job_*.json` per detected recording. We've copied them next to this notebook at `./dispatch_results/`. Each describes a serialized SpikeInterface recording the downstream preprocessing/sorting capsules consume.

In [7]:
job_files = sorted(local_results_dir.glob("job_*.json"))
print(f"Found {len(job_files)} job file(s) in {local_results_dir.name}/:")
for jf in job_files:
    print(" ", jf.name)

if job_files:
    job = json.loads(job_files[0].read_text())
    print()
    print("Top-level keys:", list(job.keys()))
    print("session_name:  ", job.get("session_name"))
    print("recording_name:", job.get("recording_name"))

Found 1 job file(s) in dispatch_results/:
  job_0.json

Top-level keys: ['session_name', 'recording_name', 'recording_dict', 'skip_times', 'duration', 'input_folder', 'debug', 'multi_input']
session_name:   session0
recording_name: block0_None_recording1
